# Model check: can we run all three translators?

Phase 1 of our plan, run before we wrote the test sentences. This notebook checks that the three models load in a fresh Colab with no sign-in, how long they take, and whether NLLB's top 8 options ever include the feminine or formal Serbian forms our picker needs.

**How to run:** Runtime → Change runtime type → T4 GPU (optional, but much faster), then Runtime → Run all. Downloading the models takes a few minutes the first time.

The four sentences below are placeholders for this first check. Our real 60 sentences came later (`data/test_sentences.csv`, used in `model.ipynb`).

## 1. Setup

In [1]:
# Colab already has transformers and torch. These two extra libraries are what the
# translation models use to split text into word pieces.
%pip install -q sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time, random
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Use the GPU if Colab gave us one; otherwise the CPU (slower, still works).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cpu


## 2. Test sentences (placeholders)

Each one has an English meaning and the labels a human would add: who is speaking and to whom.

In [3]:
sentences = pd.DataFrame([
    {"mandarin": "我检查过了。",        "english_meaning": "I checked it.",                    "speaker": "woman", "listener": "man",     "formal": False},
    {"mandarin": "您昨天来上班了吗？",  "english_meaning": "Did you come to work yesterday?",  "speaker": "man",   "listener": "manager", "formal": True},
    {"mandarin": "你昨天来上班了吗？",  "english_meaning": "Did you come to work yesterday?",  "speaker": "man",   "listener": "woman",   "formal": False},
    {"mandarin": "我已经把机器关了。",  "english_meaning": "I already turned the machine off.", "speaker": "woman", "listener": "manager", "formal": True},
])
sentences

,mandarin,english_meaning,speaker,listener,formal
0,我检查过了。,I checked it.,woman,man,False
1,您昨天来上班了吗？,Did you come to work yesterday?,man,manager,True
2,你昨天来上班了吗？,Did you come to work yesterday?,man,woman,False
3,我已经把机器关了。,I already turned the machine off.,woman,manager,True


## 3. Load the models

- `Helsinki-NLP/opus-mt-zh-en`: Mandarin → English (CC BY 4.0)
- `Helsinki-NLP/opus-mt-tc-base-en-sh`: English → Serbian, Croatian or Bosnian; we ask for Serbian in Cyrillic (CC BY 4.0)
- `facebook/nllb-200-distilled-600M`: translates directly between 200 languages (CC BY-NC 4.0, non-commercial)

None of them need a Hugging Face account.

In [4]:
def load(name, **tok_kwargs):
    start = time.time()
    tok = AutoTokenizer.from_pretrained(name, **tok_kwargs)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
    print(f"Loaded {name} in {time.time() - start:.0f} s")
    return tok, model

zh_en_tok, zh_en = load("Helsinki-NLP/opus-mt-zh-en")
en_sr_tok, en_sr = load("Helsinki-NLP/opus-mt-tc-base-en-sh")
nllb_tok, nllb = load("facebook/nllb-200-distilled-600M", src_lang="zho_Hans")

Loaded Helsinki-NLP/opus-mt-zh-en in 1 s


Loaded Helsinki-NLP/opus-mt-tc-base-en-sh in 1 s


Loaded facebook/nllb-200-distilled-600M in 2 s


## 4. Translate

`translate()` runs one model on a list of sentences. `n_options` asks for several translations per sentence (beam search keeps the best few guesses and returns them, most likely first).

In [5]:
def translate(tok, model, texts, n_options=1, **gen_kwargs):
    batch = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**batch, num_beams=max(4, n_options), num_return_sequences=n_options,
                             max_new_tokens=64, **gen_kwargs)
    decoded = tok.batch_decode(out, skip_special_tokens=True)
    # group the flat list back into n_options per input sentence
    return [decoded[i * n_options:(i + 1) * n_options] for i in range(len(texts))]

mandarin = sentences["mandarin"].tolist()
timings = {}

# Route 1: Mandarin -> English -> Serbian (what many apps do)
start = time.time()
english = [o[0] for o in translate(zh_en_tok, zh_en, mandarin)]
via_english = [o[0] for o in translate(en_sr_tok, en_sr, [">>srp_Cyrl<< " + e for e in english])]
timings["through English"] = time.time() - start

# Route 2: Mandarin -> Serbian directly with NLLB, top choice only
SERBIAN = nllb_tok.convert_tokens_to_ids("srp_Cyrl")
start = time.time()
direct = [o[0] for o in translate(nllb_tok, nllb, mandarin, forced_bos_token_id=SERBIAN)]
timings["direct (NLLB)"] = time.time() - start

# NLLB's top 8 options, which the picker will choose from
start = time.time()
options8 = translate(nllb_tok, nllb, mandarin, n_options=8, forced_bos_token_id=SERBIAN)
timings["NLLB top 8"] = time.time() - start

results = sentences[["mandarin", "speaker", "listener", "formal"]].copy()
results["model English"] = english
results["through English"] = via_english
results["direct (NLLB)"] = direct
pd.set_option("display.max_colwidth", None)
results

,mandarin,speaker,listener,formal,model English,through English,direct (NLLB)
0,我检查过了。,woman,man,False,I checked.,Проверио сам.,Проверио сам.
1,您昨天来上班了吗？,man,manager,True,Did you come to work yesterday?,Да ли сте дошли на посао јуче?,Да ли сте дошли на посао јуче?
2,你昨天来上班了吗？,man,woman,False,Did you come to work yesterday?,Да ли сте дошли на посао јуче?,Да ли си јуче дошао на посао?
3,我已经把机器关了。,woman,manager,True,I've turned the machine off.,Искључио сам машину.,Ја сам га искључио.


## 5. Do the 8 options include the forms we need?

A rough first look, before our Serbian speaker helped us write the real rules: does any option use a feminine past tense (a word ending in -ла next to сам), or the formal сте / Ви?

In [6]:
import re

def has_feminine_i(s):  return bool(re.search(r"\w+ла сам|сам \w+ла", s))
def has_formal_you(s):  return bool(re.search(r"\bсте\b|\bВи\b|\bВас\b", s))

for row, opts in zip(sentences.itertuples(), options8):
    print(f"\n{row.mandarin}  (speaker: {row.speaker}, formal: {row.formal})")
    for i, o in enumerate(opts, 1):
        tags = [t for t, ok in [("feminine I", has_feminine_i(o)), ("formal you", has_formal_you(o))] if ok]
        print(f"  {i}. {o}" + (f"   <- {', '.join(tags)}" if tags else ""))


我检查过了。  (speaker: woman, formal: False)
  1. Проверио сам.
  2. Погледао сам.
  3. Проверил сам.
  4. Проверила сам.   <- feminine I
  5. Проверио сам.
  6. Ја сам проверила.   <- feminine I
  7. Проверив сам.
  8. Проверена сам.

您昨天来上班了吗？  (speaker: man, formal: True)
  1. Да ли сте дошли на посао јуче?   <- formal you
  2. Да ли си јуче дошао на посао?
  3. Јесте ли дошли на посао јуче?
  4. Да ли сте дошли на посао вчера?   <- formal you
  5. Јесте ли дошли на посао вчера?
  6. Да ли сте дошли вчера на посао?   <- formal you
  7. Да ли си вчера дошао на посао?
  8. Да ли си дошао на посао јуче?

你昨天来上班了吗？  (speaker: man, formal: False)
  1. Да ли си јуче дошао на посао?
  2. Јесте ли дошли на посао јуче?
  3. Да ли си вчера дошао на посао?
  4. Да ли си дошао на посао јуче?
  5. Да ли сте дошли на посао јуче?   <- formal you
  6. Јесте ли дошли на посао вчера?
  7. Да ли си јуче био на послу?
  8. Да ли си дошао на посао вчера?

我已经把机器关了。  (speaker: woman, formal: True)
  1. Ја са

## 6. Timing

In [7]:
for name, secs in timings.items():
    print(f"{name:16s} {secs:6.1f} s for {len(mandarin)} sentences ({secs / len(mandarin):.2f} s each)")

through English     0.4 s for 4 sentences (0.09 s each)
direct (NLLB)       2.7 s for 4 sentences (0.69 s each)
NLLB top 8          0.7 s for 4 sentences (0.18 s each)


## What to check before saying "go"

- Did all three models load with no sign-in?
- Is the Serbian in Cyrillic, and does it read naturally? (our Serbian speaker checks)
- Do the 8 options ever include the feminine or formal forms? If they never do, the picker has nothing to pick from, and we rethink.
- Is the speed OK for 60-80 sentences?

Credits: models by Helsinki-NLP (OPUS-MT, CC BY 4.0) and Meta (NLLB-200, CC BY-NC 4.0), loaded with Hugging Face transformers.